# 03 · Paired-DRR validation and geometry calibration (Tests A/B/C)

> **Research use only.** Generated radiographs and CT volumes are synthetic and patient-conditioned. They are
> estimates, not acquisitions of the patient and not clinical ground truth. Never use them for diagnosis or treatment.
> Upload only images you are authorised to use, and remove burned-in identifiers first.

Ground truth comes from **CT volumes**. Each CT gives a ground-truth volume, display-oriented reference DRRs (NumPy),
SV-DRR-geometry DRRs (PA, AP-like and lateral), and DVG-projector views. Each test arm changes one variable at a time:

| Arm | slot 0 | slot 1 | isolates |
|---|---|---|---|
| A_native | DVG projector frontal | DVG projector lateral | DVG ceiling on its own domain |
| C_svdrr | SV-DRR PA | true SV-DRR lateral | DRR style / field-of-view mismatch (vs A) |
| B_synth | SV-DRR PA | **Parallax lateral** | synthesis error (vs C) |
| B_ap | SV-DRR AP-like | **Parallax lateral from AP** | PA-vs-AP sensitivity (vs B) |
| C_ap | SV-DRR AP-like | true SV-DRR lateral | AP frontal effect with a true lateral |

The notebook also **calibrates** the uncalibrated conventions in `PipelineConfig`: the Parallax lateral azimuth sign,
the Parallax output orientation, and the DVG slot transforms. It writes them to
`OUTPUT_ROOT/calibration/geometry_calibration.json`, which notebook 04 loads.

**Data authorisation.** Use only CTs you are authorised to process. LIDC-IDRI is **contaminated** for this purpose:
all 16 SV-DRR validation patients are in the X2CT/DVG training split (`validation_data/lidc_split/`). The cleanest
options are your own authorised local CTs, or a public non-LIDC chest CT collection (e.g. TCIA NSCLC-Radiomics).

In [ ]:
#@title 1. Configuration { display-mode: "form" }
REPO_URL = "https://github.com/sanafathima-00/RadioVue.git"  #@param {type:"string"}
REPO_BRANCH = "claude/single-xray-vlm-pipeline"  #@param {type:"string"}
#@markdown If the pipeline code is not pushed yet, upload `radiovue_colab_bundle.zip` (see SETUP_GUIDE.md) to this Drive path:
CODE_BUNDLE_ON_DRIVE = "RadioVue/code/radiovue_colab_bundle.zip"  #@param {type:"string"}
DRIVE_ROOT = "/content/drive/MyDrive/RadioVue"  #@param {type:"string"}
DVG_CHECKPOINTS_ON_DRIVE = DRIVE_ROOT + "/Volumea/models/DVG-Diffusion"
PARALLAX_MODEL_ON_DRIVE = DRIVE_ROOT + "/Parallax/models/DiT-fb-512"  # used if present, else downloaded from Hugging Face
OUTPUT_ROOT = DRIVE_ROOT + "/outputs/single_xray_pipeline"
REPO_DIR = "/content/RadioVue"

In [ ]:
#@title 2. GPU, Drive and code
import os, sys, subprocess, zipfile, importlib.util
from pathlib import Path

smi = subprocess.run(["nvidia-smi"], capture_output=True, text=True)
print(smi.stdout[:700] or smi.stderr)
assert smi.returncode == 0, "No GPU. Runtime > Change runtime type > GPU (L4/A100 recommended, T4 minimum)."

from google.colab import drive
drive.mount("/content/drive")

bundle = Path("/content/drive/MyDrive") / CODE_BUNDLE_ON_DRIVE
if not Path(REPO_DIR, "radiovue_pipeline").exists():
    if bundle.is_file():
        Path(REPO_DIR).mkdir(parents=True, exist_ok=True)
        zipfile.ZipFile(bundle).extractall(REPO_DIR)
        print("Code from Drive bundle:", bundle)
    else:
        subprocess.run(["git", "clone", "--depth", "1", "-b", REPO_BRANCH, REPO_URL, REPO_DIR], check=True)
assert Path(REPO_DIR, "radiovue_pipeline").exists(), (
    "radiovue_pipeline/ not found. Push the branch that contains it, or upload the code bundle (SETUP_GUIDE.md).")
os.chdir(REPO_DIR)
sys.path.insert(0, REPO_DIR)
rev = subprocess.run(["git", "rev-parse", "HEAD"], capture_output=True, text=True).stdout.strip()
print("Code revision:", rev or "(bundle, see BUNDLE_MANIFEST.json)")

# colab_setup has no third-party imports, so it can run before packages are installed.
spec = importlib.util.spec_from_file_location("colab_setup", f"{REPO_DIR}/radiovue_pipeline/colab_setup.py")
cs = importlib.util.module_from_spec(spec); spec.loader.exec_module(cs)
GPU = cs.gpu_report()

In [ ]:
#@title 3. Install dependencies and stage models (first run ~10-15 min)
cs.install_runtime(parallax=True, dvg=True, tools=True)
cs.build_drr_projector(f"{REPO_DIR}/Volumea")
DVG_CKPT_DIR = cs.stage_dvg_checkpoints(DVG_CHECKPOINTS_ON_DRIVE, verify_hash=False)
cs.ensure_parallax_shims(f"{REPO_DIR}/Parallax")
PARALLAX_MODEL_DIR = cs.stage_parallax_model(PARALLAX_MODEL_ON_DRIVE)

In [ ]:
#@title 4. Cases (DICOM CT series folders on Drive)
DATA_AUTHORISED = False  #@param {type:"boolean"}
CT_ROOT = DRIVE_ROOT + "/validation_ct"  #@param {type:"string"}
MAX_CASES = 8  #@param {type:"integer"}
SEEDS = [0, 1, 2, 3]
assert DATA_AUTHORISED, "Confirm you are authorised to process these CT series, then tick DATA_AUTHORISED."
case_dirs = sorted(p for p in Path(CT_ROOT).iterdir() if p.is_dir())[:MAX_CASES]
print(len(case_dirs), "case folders (names are not printed)")
VAL = Path(OUTPUT_ROOT) / "validation"; VAL.mkdir(parents=True, exist_ok=True)

In [ ]:
#@title 5. CT -> ground-truth volume, reference DRRs, SV-DRR DRRs (PA, AP-like, lateral +/-90)
import json, numpy as np, torch
from radiovue_pipeline import drr, calibration as cal
cases = {}
for i, d in enumerate(case_dirs):
    cid = f"case{i:02d}"
    ct = drr.load_dicom_ct_series(d)
    iso = drr.resample_isotropic(ct.hu, ct.spacing_zyx, 2.5)
    cube = drr.center_crop_or_pad(iso, 128, -1024.0)
    gt01 = (np.clip(cube + 1024.0, 0, 2500) / 2500).astype(np.float32)
    nii = VAL / f"{cid}_hu.nii.gz"; drr.write_nifti(cube, (2.5, 2.5, 2.5), nii)
    refs = cal.reference_views(cube)
    pa_r, ap_r = drr.SvdrrDrrRenderer(nii, 512, "PA"), drr.SvdrrDrrRenderer(nii, 512, "AP", sdd=1000.0, source_to_isocenter=850.0)
    raw = {"pa": pa_r.render(0), "ap": ap_r.render(0), "lat_p90": pa_r.render(90), "lat_m90": pa_r.render(-90)}
    fits = {"pa": cal.best_dihedral_match(raw["pa"], refs["frontal"]), "ap": cal.best_dihedral_match(raw["ap"], refs["frontal"]),
            "lat_p90": cal.best_dihedral_match(raw["lat_p90"], refs["lateral"]), "lat_m90": cal.best_dihedral_match(raw["lat_m90"], refs["lateral"])}
    disp = {k: cal.apply_transform(v, fits[k]["model_to_display"]) for k, v in raw.items()}
    cases[cid] = {"cube": cube, "gt01": gt01, "refs": refs, "display": disp, "svdrr_fits": fits, "meta": ct.metadata}
    print(cid, {k: (f["model_to_display"], round(f["ncc"], 3), round(f["margin_over_runner_up"], 3)) for k, f in fits.items()})

In [ ]:
#@title 6. Parallax synthesis from display-oriented PA and AP DRRs at both azimuth signs (then free GPU)
from PIL import Image
from radiovue_pipeline.view_synthesis import ParallaxSynthesizer
from radiovue_pipeline.io_utils import from_float01, to_float01
synth = ParallaxSynthesizer(f"{REPO_DIR}/Parallax", PARALLAX_MODEL_DIR, 512)
for cid, c in cases.items():
    c["synth"] = {}
    for src in ("pa", "ap"):
        for az in (90, -90):
            outs = synth.generate(from_float01(c["display"][src]), az, SEEDS)
            c["synth"][(src, az)] = [to_float01(o) for o in outs]
    print(cid, "done")
del synth; torch.cuda.empty_cache()

In [ ]:
#@title 7. Calibrate Parallax: which azimuth gives a lateral, and which transform maps its output to display orientation
votes = {}
for cid, c in cases.items():
    for az in (90, -90):
        f = cal.best_dihedral_match(c["synth"][("pa", az)][0], c["refs"]["lateral"])
        votes.setdefault(az, []).append(f)
summary = {az: {"mean_ncc": float(np.mean([f["ncc"] for f in fs])),
                "transforms": [f["model_to_display"] for f in fs]} for az, fs in votes.items()}
best_az = max(summary, key=lambda a: summary[a]["mean_ncc"])
transforms = summary[best_az]["transforms"]
best_tf = max(set(transforms), key=transforms.count)
consistent = transforms.count(best_tf) == len(transforms)
print(json.dumps(summary, indent=1)); print("Chosen azimuth:", best_az, "| output->display:", best_tf, "| consistent across cases:", consistent)
for c in cases.values():
    c["synth_pa"] = [cal.apply_transform(s, best_tf) for s in c["synth"][("pa", best_az)]]
    c["synth_ap"] = [cal.apply_transform(s, best_tf) for s in c["synth"][("ap", best_az)]]
    c["gt_lateral"] = c["display"]["lat_p90"] if c["svdrr_fits"]["lat_p90"]["ncc"] >= c["svdrr_fits"]["lat_m90"]["ncc"] else c["display"]["lat_m90"]

In [ ]:
#@title 8. Load Volumea, find DVG volume orientation (coarse self-consistency), run arms A/B/C
from radiovue_pipeline.volumea_adapter import VolumeaReconstructor
from radiovue_pipeline import validation as V
from radiovue_pipeline.metrics import LPIPSMetric
recon = VolumeaReconstructor(f"{REPO_DIR}/Volumea", DVG_CKPT_DIR)
first = next(iter(cases.values()))
ranking = V.search_dvg_volume_orientation(recon.model, first["gt01"], f"{REPO_DIR}/Volumea")
orientation = ranking[0]
print("Top orientations:", [(r["perm"], r["flips"], round(r["coarse_ncc"], 3)) for r in ranking[:4]])
lp = LPIPSMetric()
results = {}
for cid, c in cases.items():
    views = {"pa": c["display"]["pa"], "ap": c["display"]["ap"], "lateral": c["gt_lateral"]}
    r = V.run_case(cid, c["cube"], c["gt01"], orientation, views, c["synth_pa"], c["synth_ap"], recon, f"{REPO_DIR}/Volumea", lp)
    np.savez_compressed(VAL / f"{cid}_volumes.npz", **r.pop("volumes"))
    results[cid] = r
    print(cid, {arm: round(m["ssim_3d"], 3) for arm, m in r["arms"].items()})

In [ ]:
#@title 9. Aggregate metrics, patient specificity, QC calibration; write calibration JSON
import pandas as pd
from radiovue_pipeline.metrics import patient_specificity
from radiovue_pipeline.consistency import row_profile_consistency
rows = [{"case": cid, "arm": arm, **{k: v for k, v in m.items() if k != "note"}} for cid, r in results.items() for arm, m in r["arms"].items()]
arm_table = pd.DataFrame(rows); arm_table.to_csv(VAL / "ct_arm_metrics.csv", index=False)
display(arm_table.groupby("arm").agg(["mean", "std"]).round(4))
syn = pd.DataFrame([{"case": cid, "input": k, **r["synthesis_2d"][k]} for cid, r in results.items() for k in ("from_pa", "from_ap")])
syn.to_csv(VAL / "synthesis_2d_metrics.csv", index=False); display(syn.groupby("input").mean(numeric_only=True).round(4))
ids = list(cases)
spec = patient_specificity([cases[i]["synth_pa"][0] for i in ids], [cases[i]["gt_lateral"] for i in ids])
print("Patient specificity top-1:", spec["top1_identification"], "chance:", spec["chance"])
qc_true = [row_profile_consistency(cases[i]["display"]["pa"], cases[i]["synth_pa"][0])["row_profile_corr"] for i in ids]
qc_gt = [row_profile_consistency(cases[i]["display"]["pa"], cases[i]["gt_lateral"])["row_profile_corr"] for i in ids]
qc_mismatch = [row_profile_consistency(cases[i]["display"]["pa"], cases[j]["gt_lateral"])["row_profile_corr"] for i in ids for j in ids if i != j]
print("Row-profile corr  PA~synthetic:", np.round(qc_true, 3), " PA~true lateral:", np.round(qc_gt, 3),
      " PA~other patient's lateral (mean):", round(float(np.mean(qc_mismatch)), 3) if qc_mismatch else None)
dvg_front = [r["calibration"]["dvg_frontal"]["display_to_model"] for r in results.values()]
dvg_lat = [r["calibration"]["dvg_lateral"]["display_to_model"] for r in results.values()]
calib = {
    "lateral_azimuth_deg": float(best_az), "lateral_azimuth_calibrated": bool(consistent),
    "parallax_output_transform": best_tf,
    "dvg_frontal_transform": max(set(dvg_front), key=dvg_front.count),
    "dvg_lateral_transform": max(set(dvg_lat), key=dvg_lat.count),
    "dvg_geometry_calibrated": len(set(dvg_front)) == 1 and len(set(dvg_lat)) == 1,
    "dvg_volume_orientation": {"perm": list(orientation["perm"]), "flips": list(orientation["flips"]), "coarse_ncc": orientation["coarse_ncc"]},
    "evidence": {"n_cases": len(ids), "parallax_votes": summary, "patient_specificity": spec,
                 "row_profile": {"pa_vs_synthetic": qc_true, "pa_vs_true_lateral": qc_gt, "pa_vs_other_lateral_mean": float(np.mean(qc_mismatch)) if qc_mismatch else None}},
}
(Path(OUTPUT_ROOT) / "calibration").mkdir(parents=True, exist_ok=True)
(Path(OUTPUT_ROOT) / "calibration" / "geometry_calibration.json").write_text(json.dumps(calib, indent=2, default=str))
(VAL / "full_results.json").write_text(json.dumps(results, indent=2, default=str))
print(json.dumps({k: v for k, v in calib.items() if k != "evidence"}, indent=1))

In [ ]:
#@title 10. Visual comparison for one case (all synthetic panels labelled)
import matplotlib.pyplot as plt
cid = ids[0]; c = cases[cid]; vols = np.load(VAL / f"{cid}_volumes.npz")
fig, ax = plt.subplots(2, 6, figsize=(24, 8))
panels = [("PA DRR (input)", c["display"]["pa"]), ("True lateral DRR", c["gt_lateral"]), ("SYNTHETIC lateral (PA)", c["synth_pa"][0]),
          ("SYNTHETIC lateral (AP)", c["synth_ap"][0]), ("Seed std (uncertainty)", np.std(np.stack(c["synth_pa"]), 0)), ("AP-like DRR", c["display"]["ap"])]
for a, (t, im) in zip(ax[0], panels): a.imshow(im, cmap="gray"); a.set_title(t); a.axis("off")
for a, k in zip(ax[1], ["ground_truth", "A_native", "C_svdrr", "B_synth", "B_ap", "C_ap"]):
    a.imshow(vols[k][:, 64, :], cmap="gray", vmin=0, vmax=1); a.set_title(k if k == "ground_truth" else f"SYNTHETIC CT: {k}"); a.axis("off")
plt.tight_layout(); plt.savefig(VAL / f"{cid}_comparison.png", dpi=110); plt.show()